In [4]:
import requests
import pandas as pd
import numpy as np
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import os

load_dotenv()
user_agent = os.getenv("USER_AGENT")
print("User Agent:", user_agent)

def get_title(soup):
    try:
        return soup.find("span", id="productTitle").get_text(strip=True)
    except AttributeError:
        return ""


def get_price(soup):
    try:
        return soup.find("span", class_="a-offscreen").get_text(strip=True)
    except AttributeError:
        return ""


def get_rating(soup):
    try:
        return soup.find("span", class_="a-icon-alt").get_text(strip=True)
    except AttributeError:
        return ""


def get_review_count(soup):
    try:
        return soup.find(
            "span",
            id="acrCustomerReviewText"
        ).get_text(strip=True)
    except AttributeError:
        return ""


def get_availability(soup):
    try:
        return soup.find(
            "div",
            id="availability"
        ).get_text(strip=True)
    except AttributeError:
        return ""


if __name__ == "__main__":

    # ---------------- HEADERS ----------------
    HEADERS = {
        "User-Agent": user_agent,
        "Accept-Language": "en-US,en;q=0.5"
    }

    # ---------------- AMAZON URL ----------------
    URL = "https://www.amazon.com/s?k=playstation+4"

    # ---------------- GET SEARCH PAGE ----------------
    webpage = requests.get(
        URL,
        headers=HEADERS,
        timeout=10
    )

    print("Status Code:", webpage.status_code)

    soup = BeautifulSoup(
        webpage.content,
        "html.parser"
    )

    # ---------------- GET PRODUCT LINKS ----------------
    links = soup.find_all(
        "a",
        class_="a-link-normal s-no-outline"
    )

    print("Links Found:", len(links))

    links_list = []

    for link in links:
        href = link.get("href")

        if href:
            links_list.append(href)

    # ---------------- DATA DICTIONARY ----------------
    d = {
        "title": [],
        "price": [],
        "rating": [],
        "reviews": [],
        "availability": [],
        "link": []
    }

    # ---------------- PRODUCT LOOP ----------------
    for link in links_list:

        product_url = "https://www.amazon.com" + link

        try:
            new_webpage = requests.get(
                product_url,
                headers=HEADERS,
                timeout=10
            )

            print(
                "Product Status:",
                new_webpage.status_code
            )

            new_soup = BeautifulSoup(
                new_webpage.content,
                "html.parser"
            )

            d["title"].append(get_title(new_soup))
            d["price"].append(get_price(new_soup))
            d["rating"].append(get_rating(new_soup))
            d["reviews"].append(get_review_count(new_soup))
            d["availability"].append(get_availability(new_soup))
            d["link"].append(product_url)

        except requests.RequestException as e:
            print("Request failed:", e)

    # ---------------- CREATE DATAFRAME ----------------
    amazon_df = pd.DataFrame(d)

    # Replace empty titles with NaN
    amazon_df["title"] = amazon_df["title"].replace(
        "",
        np.nan
    )

    # Remove products without title
    amazon_df = amazon_df.dropna(
        subset=["title"]
    )

    # ---------------- SAVE CSV ----------------
    amazon_df.to_csv(
        "amazon_data.csv",
        index=False
    )

    print("\nFinal Data:")
    print(amazon_df)


    print("Status:", webpage.status_code)
    print("Links found:", len(links))
    print("Links list:", len(links_list))

User Agent: Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/153.0.0.0 Safari/537.36
Status Code: 200
Links Found: 18
Product Status: 200
Product Status: 200
Product Status: 200
Product Status: 200
Product Status: 200
Product Status: 200
Product Status: 200
Product Status: 200
Product Status: 200
Product Status: 200
Product Status: 200
Product Status: 200
Product Status: 200
Product Status: 200
Product Status: 200
Product Status: 200
Request failed: HTTPSConnectionPool(host='www.amazon.comhttps', port=443): Max retries exceeded with url: //aax-us-east-retail-direct.amazon.com/x/c/JHktXtWkTDvIzIA_yI8BalIAAAGhG20UnAEAAAH2AQBvbm9fdHhuX2JpZDEgICBvbm9fdHhuX2ltcDIgICDpGhtf/clv1c_ek-wBPfTJcbUrRdbVIfEL2YKkKsKKGBUdKRFRik5HpvhAw2NzVWdKuS0xQh5Uef2VuR4IpF8rdirE2DFBWHPSsRm7YGistgLoSTyIiOd1-bwqyH1PbQFXF0B4vc4pnArxfBXINiajSm8ia0chUPxMk6EltR81lbi2uTwBbWt0LRfyWoRKPgNVbRGQF5uN9tLnOaOz5ruGKvD1Kg9lgvCZWSB3Z8JdHbzkodX-fiseQq0QEa0mMJmT-2yfxqpfMJPwiqduj45C7tSYKrCa0dxVAx

In [3]:
df = pd.read_csv("amazon_data.csv")
print(df.head())

                                               title        price  \
0                     PlayStation 4 Slim 1TB Console  INR1,391.80   
1  CRKD Gibson SG Black 20th Anniversary Edition ...  INR9,193.80   
2  PDP Riffmaster Wireless Guitar Controller for ...  INR8,274.32   
3  AceGamer 2 Pack for PS4 Controller, Hall Effec...   INR1,45084   
4  OIVO PS4 Stand for PlayStation 4 Console, Cool...  INR16,83900   

               rating   reviews  \
0  4.7 out of 5 stars  (15,966)   
1  4.6 out of 5 stars      (62)   
2  4.5 out of 5 stars   (1,194)   
3  4.2 out of 5 stars  (10,642)   
4  4.5 out of 5 stars  (47,689)   

                                        availability  \
0  This item cannot be shipped to your selected d...   
1                                           In Stock   
2                                           In Stock   
3  This item cannot be shipped to your selected d...   
4  This item cannot be shipped to your selected d...   

                                     

In [14]:
amazon_df

,title,price,rating,reviews,availability,link
0,PlayStation 4 Slim 1TB Console,"INR1,391.80",4.7 out of 5 stars,"(15,966)",This item cannot be shipped to your selected d...,https://www.amazon.com/PlayStation-4-Slim-1TB-...
1,CRKD Gibson SG Black 20th Anniversary Edition ...,"INR9,193.80",4.6 out of 5 stars,(62),In Stock,https://www.amazon.com/sspa/click?ie=UTF8&spc=...
2,PDP Riffmaster Wireless Guitar Controller for ...,"INR8,274.32",4.5 out of 5 stars,"(1,194)",In Stock,https://www.amazon.com/sspa/click?ie=UTF8&spc=...
3,"AceGamer 2 Pack for PS4 Controller, Hall Effec...","INR1,45084",4.2 out of 5 stars,"(10,642)",This item cannot be shipped to your selected d...,https://www.amazon.com/sspa/click?ie=UTF8&spc=...
4,"OIVO PS4 Stand for PlayStation 4 Console, Cool...","INR16,93289",4.5 out of 5 stars,"(47,689)",This item cannot be shipped to your selected d...,https://www.amazon.com/sspa/click?ie=UTF8&spc=...


In [5]:
import pandas as pd

df = pd.read_csv("amazon_data.csv")
print(df.head())

                                               title         price  \
0                     PlayStation 4 Slim 1TB Console   INR1,391.80   
1              PlayStation®5 Digital Edition – 825GB    INR5,32232   
2  New Sony PlayStation 4 Pro 1TB Red Dead Redemp...           NaN   
3                                      PlayStation 4    INR1,69280   
4                      PlayStation 4 Pro 1TB Console  INR24,187.07   

               rating   reviews  \
0  4.7 out of 5 stars  (15,970)   
1  4.6 out of 5 stars     (998)   
2  4.0 out of 5 stars     (101)   
3  4.4 out of 5 stars  (13,867)   
4  4.7 out of 5 stars  (12,792)   

                                        availability  \
0  This item cannot be shipped to your selected d...   
1  This item cannot be shipped to your selected d...   
2                   Usually ships within 6 to 7 days   
3  This item cannot be shipped to your selected d...   
4                   Usually ships within 5 to 6 days   

                               